# Train Custom "Hey Vox" Wake Word Model

Uses openwakeword's **official training pipeline** (`train.py` + YAML config) with:
- Piper TTS for synthetic positive generation (50K+ clips)
- Adversarial negative generation (phoneme-similar phrases)
- Room impulse response (RIR) augmentation
- Background noise mixing
- ACAV100M features for general negative data
- HuggingFace validation features for FP rate calibration

**License-safe**: Uses MIT RIRs, CC-licensed background audio, and CC-BY ACAV100M features.

**Output**: `hey_vox.onnx` compatible with `openwakeword.Model()`

---

## Before you start

1. Use **GPU runtime**: Runtime → Change runtime type → T4 GPU
2. Upload `personal_recordings.zip` (75 real "hey vox" clips) when prompted
3. Training takes ~45-90 minutes on T4

## 1. Install Dependencies & Apply Patches

In [ ]:
# Install first, then patch — pip install may upgrade torch, losing patches
!pip install -q openwakeword soundfile scipy datasets huggingface_hub
!pip install -q onnx onnxruntime
print('Dependencies installed')

In [ ]:
# ── PyTorch 2.6 compatibility patches ──────────────────────────────────
# Applied AFTER pip install to survive any torch upgrades
import torch
_original_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _original_load(*args, **kwargs)
torch.load = _patched_load
print(f'Patched torch.load (PyTorch {torch.__version__})')

# torchaudio.set_audio_backend is removed in modern torchaudio
import torchaudio
if not hasattr(torchaudio, 'set_audio_backend'):
    torchaudio.set_audio_backend = lambda x: None
    print('Stubbed torchaudio.set_audio_backend')

print('Patches applied')

## 2. Clone Piper Sample Generator

In [ ]:
import os

PIPER_DIR = '/content/piper-sample-generator'
if not os.path.exists(PIPER_DIR):
    !git clone https://github.com/dscripka/piper-sample-generator.git {PIPER_DIR}
    !cd {PIPER_DIR} && pip install -q -e .
    print('Piper sample generator cloned and installed')
else:
    print('Piper already exists')

## 3. Download Training Resources

- **MIT Room Impulse Responses** — for reverb augmentation
- **Background audio clips** — ambient noise for augmentation
- **ACAV100M features** — large-scale negative data (pre-computed)
- **Validation features** — for false-positive rate calibration

In [ ]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

OUTPUT_DIR = '/content/hey_vox_training'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── MIT Room Impulse Responses ──────────────────────────────────────────
RIR_DIR = os.path.join(OUTPUT_DIR, 'mit_rirs')
if not os.path.exists(RIR_DIR) or len(list(Path(RIR_DIR).rglob('*.wav'))) == 0:
    os.makedirs(RIR_DIR, exist_ok=True)
    print('Downloading MIT RIRs...')
    rir_zip = hf_hub_download('davidscripka/openwakeword_features', 'mit_rirs.zip', local_dir='/tmp/hf_downloads')
    !unzip -q -o {rir_zip} -d {RIR_DIR}
    n_rirs = len(list(Path(RIR_DIR).rglob('*.wav')))
    print(f'{n_rirs} RIR files downloaded')
else:
    print(f'MIT RIRs already downloaded ({len(list(Path(RIR_DIR).rglob("*.wav")))} files)')

# ── Background noise clips ──────────────────────────────────────────────
BG_DIR = os.path.join(OUTPUT_DIR, 'background_clips')
if not os.path.exists(BG_DIR) or len(list(Path(BG_DIR).rglob('*.wav'))) == 0:
    os.makedirs(BG_DIR, exist_ok=True)
    print('Downloading background audio...')
    bg_zip = hf_hub_download('davidscripka/openwakeword_features', 'background_clips.zip', local_dir='/tmp/hf_downloads')
    !unzip -q -o {bg_zip} -d {BG_DIR}
    n_bg = len(list(Path(BG_DIR).rglob('*.wav')))
    print(f'{n_bg} background clips downloaded')
else:
    print(f'Background clips already downloaded ({len(list(Path(BG_DIR).rglob("*.wav")))} files)')

# Verify and show actual directory structure
print('\nRIR dirs with WAV files:')
for d in sorted(set(p.parent for p in Path(RIR_DIR).rglob('*.wav'))):
    print(f'  {d} ({len(list(d.glob("*.wav")))} files)')
print('BG dirs with WAV files:')
for d in sorted(set(p.parent for p in Path(BG_DIR).rglob('*.wav'))):
    print(f'  {d} ({len(list(d.glob("*.wav")))} files)')

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

# ── ACAV100M pre-computed features (large negative dataset) ─────────────
ACAV_FEATURES = os.path.join(OUTPUT_DIR, 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy')
if not os.path.exists(ACAV_FEATURES):
    print('Downloading ACAV100M features (~4GB, takes a few minutes)...')
    downloaded = hf_hub_download('davidscripka/openwakeword_features', 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy', local_dir=OUTPUT_DIR)
    # hf_hub_download may return a cache path — copy to expected location if needed
    if not os.path.exists(ACAV_FEATURES):
        if os.path.exists(downloaded):
            shutil.move(downloaded, ACAV_FEATURES)
            print(f'Moved from {downloaded}')
        else:
            candidates = list(Path(OUTPUT_DIR).rglob('*ACAV100M*.npy'))
            if candidates:
                shutil.move(str(candidates[0]), ACAV_FEATURES)
                print(f'Moved from {candidates[0]}')
    print(f'ACAV100M features: {os.path.getsize(ACAV_FEATURES) / 1e9:.1f} GB')
else:
    print(f'ACAV100M features already exist ({os.path.getsize(ACAV_FEATURES) / 1e9:.1f} GB)')

# ── Validation features for FP rate calibration ─────────────────────────
VAL_FEATURES = os.path.join(OUTPUT_DIR, 'validation_set_features.npy')
if not os.path.exists(VAL_FEATURES):
    print('Downloading validation features...')
    downloaded = hf_hub_download('davidscripka/openwakeword_features', 'validation_set_features.npy', local_dir=OUTPUT_DIR)
    if not os.path.exists(VAL_FEATURES):
        if os.path.exists(downloaded):
            shutil.move(downloaded, VAL_FEATURES)
        else:
            candidates = list(Path(OUTPUT_DIR).rglob('validation_set_features.npy'))
            if candidates:
                shutil.move(str(candidates[0]), VAL_FEATURES)
    print(f'Validation features: {os.path.getsize(VAL_FEATURES) / 1e6:.0f} MB')
else:
    print(f'Validation features already exist ({os.path.getsize(VAL_FEATURES) / 1e6:.0f} MB)')

## 4. Upload Personal Recordings

Upload `personal_recordings.zip` (75 real "hey vox" clips recorded locally).
These will be mixed into the training data alongside synthetic samples.

In [ ]:
from google.colab import files

PERSONAL_DIR = os.path.join(OUTPUT_DIR, 'personal_positives')
os.makedirs(PERSONAL_DIR, exist_ok=True)

if len(list(Path(PERSONAL_DIR).glob('*.wav'))) == 0:
    print('Upload personal_recordings.zip (75 real "hey vox" clips):')
    uploaded = files.upload()
    for name, data in uploaded.items():
        zip_path = f'/tmp/{name}'
        with open(zip_path, 'wb') as f:
            f.write(data)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(PERSONAL_DIR)
    n_personal = len(list(Path(PERSONAL_DIR).glob('*.wav')))
    print(f'{n_personal} personal recordings uploaded')
else:
    n_personal = len(list(Path(PERSONAL_DIR).glob('*.wav')))
    print(f'{n_personal} personal recordings already exist')

## 5. Write Training Config

In [ ]:
import yaml

# Find directories that actually contain WAV files (handles nested zip extraction)
rir_wav_dirs = sorted(set(str(p.parent) for p in Path(RIR_DIR).rglob('*.wav')))
if not rir_wav_dirs:
    rir_wav_dirs = [RIR_DIR]

bg_wav_dirs = sorted(set(str(p.parent) for p in Path(BG_DIR).rglob('*.wav')))
if not bg_wav_dirs:
    bg_wav_dirs = [BG_DIR]

config = {
    'model_name': 'hey_vox',
    'target_phrase': ['hey vox'],

    # Phrases to NOT trigger on (confusables, common speech)
    'custom_negative_phrases': [
        'hey box', 'hey fox', 'hey socks', 'hey rocks', 'hey docs',
        'next box', 'gray fox', 'hey boss', 'hey bro', 'hey bob',
        'hey there', 'hey man', 'hey you', 'hey what',
        'okay', 'hey siri', 'alexa', 'okay google', 'hey google',
        'hey cortana', 'hey jarvis', 'computer',
    ],

    # Sample counts
    'n_samples': 50000,
    'n_samples_val': 5000,
    'tts_batch_size': 64,
    'augmentation_batch_size': 16,
    'augmentation_rounds': 2,

    # Paths
    'piper_sample_generator_path': PIPER_DIR,
    'output_dir': OUTPUT_DIR,
    'rir_paths': rir_wav_dirs,
    'background_paths': bg_wav_dirs,
    'background_paths_duplication_rate': [1] * len(bg_wav_dirs),

    # Pre-computed negative features (ACAV100M — 2000 hours of audio)
    'feature_data_files': {
        'ACAV100M_sample': ACAV_FEATURES,
    },

    # Batch composition per training step
    'batch_n_per_class': {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 128,
        'positive': 128,
    },

    # Validation
    'false_positive_validation_data_path': VAL_FEATURES,

    # Model architecture
    'model_type': 'dnn',
    'layer_size': 64,  # larger than default 32 — "hey vox" is a short phrase

    # Training
    'steps': 75000,
    'max_negative_weight': 2000,
    'target_false_positives_per_hour': 0.1,
}

CONFIG_PATH = os.path.join(OUTPUT_DIR, 'hey_vox_config.yaml')
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'Config written to {CONFIG_PATH}')
print(f'  Positive samples: {config["n_samples"]} synthetic + {n_personal} personal')
print(f'  Negative: ACAV100M (2000h) + {len(config["custom_negative_phrases"])} adversarial phrases')
print(f'  RIR paths: {len(rir_wav_dirs)} dirs')
print(f'  Background paths: {len(bg_wav_dirs)} dirs')
print(f'  Target FP rate: {config["target_false_positives_per_hour"]} per hour')
print(f'  Steps: {config["steps"]}')

## 6. Generate Synthetic Clips

Generates 50K positive and 50K adversarial negative synthetic clips using Piper TTS.
Takes ~20-30 minutes on T4 GPU.

In [ ]:
!python -m openwakeword.train \
    --training_config {CONFIG_PATH} \
    --generate_clips
print('\nClip generation complete')

### 6b. Mix personal recordings into training data

Copy real recordings into the positive train directory so they get augmented alongside synthetic ones.

In [ ]:
import shutil

pos_train_dir = os.path.join(OUTPUT_DIR, 'hey_vox', 'positive_train')
if not os.path.exists(pos_train_dir):
    print(f'ERROR: {pos_train_dir} does not exist!')
    print('The generate_clips step may have failed. Check output above.')
    print('Directory listing:')
    for p in sorted(Path(OUTPUT_DIR).iterdir()):
        print(f'  {p.name}/ ' if p.is_dir() else f'  {p.name}')
else:
    # Copy personal recordings with 10x duplication (they're high-value)
    personal_wavs = sorted(Path(PERSONAL_DIR).glob('*.wav'))
    n_copied = 0
    for dup in range(10):
        for wav in personal_wavs:
            dest = os.path.join(pos_train_dir, f'personal_d{dup}_{wav.name}')
            shutil.copy2(str(wav), dest)
            n_copied += 1

    total_pos = len(list(Path(pos_train_dir).glob('*.wav')))
    print(f'Copied {n_copied} personal recording copies into positive training data')
    print(f'Total positive training clips: {total_pos}')

## 7. Augment & Extract Features

Applies room impulse responses + background noise augmentation, then extracts
openwakeword embedding features. Takes ~15-30 minutes.

In [ ]:
!python -m openwakeword.train \
    --training_config {CONFIG_PATH} \
    --augment_clips
print('\nAugmentation & feature extraction complete')

## 8. Train Model

Trains the DNN classifier using openwakeword's auto-training with:
- Dynamic negative weight scheduling
- False-positive rate targeting (0.1/hour)
- Early stopping on validation metrics

Takes ~15-30 minutes.

In [ ]:
!python -m openwakeword.train \
    --training_config {CONFIG_PATH} \
    --train_model
print('\nTraining complete')

## 9. Evaluate Model

In [ ]:
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd
from openwakeword import Model

# Find the trained model — openwakeword may save to output_dir/ or output_dir/model_name/
model_path = None
for candidate in [
    os.path.join(OUTPUT_DIR, 'hey_vox.onnx'),
    os.path.join(OUTPUT_DIR, 'hey_vox', 'hey_vox.onnx'),
]:
    if os.path.exists(candidate):
        model_path = candidate
        break

if not model_path:
    candidates = sorted(Path(OUTPUT_DIR).rglob('*.onnx'))
    if candidates:
        model_path = str(candidates[0])
    else:
        print('No .onnx model found. Directory contents:')
        for p in sorted(Path(OUTPUT_DIR).rglob('*')):
            if p.is_file() and p.stat().st_size > 1000:
                print(f'  {p.relative_to(OUTPUT_DIR)} ({p.stat().st_size / 1024:.0f} KB)')
        raise FileNotFoundError('No .onnx model found in output directory')

print(f'Model: {model_path}')
print(f'Size: {os.path.getsize(model_path) / 1024:.0f} KB')

model = Model(wakeword_models=[model_path])


def wav_to_int16_16k(path):
    """Read WAV file, resample to 16kHz, return int16 array."""
    audio, sr = sf.read(str(path))
    # Convert to mono if stereo
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    # Resample to 16kHz if needed
    if sr != 16000:
        g = gcd(sr, 16000)
        audio = resample_poly(audio, 16000 // g, sr // g).astype(np.float64)
    # Convert float to int16
    if np.issubdtype(audio.dtype, np.floating):
        audio = (audio * 32767).astype(np.int16)
    return audio


def test_detection(wav_path, threshold=0.5):
    """Feed audio through model and return max score."""
    model.reset()  # Clear state between files
    audio = wav_to_int16_16k(wav_path)
    max_score = 0.0
    for i in range(0, len(audio) - 1280, 1280):
        pred = model.predict(audio[i:i+1280])
        score = list(pred.values())[0]
        if score > max_score:
            max_score = score
    return max_score


# ── Test on personal recordings (should detect) ──────────────────────
print('\n--- Personal recordings (expect DETECT) ---')
personal_wavs = sorted(Path(PERSONAL_DIR).glob('*.wav'))[:20]
tp = 0
for wav_path in personal_wavs:
    score = test_detection(wav_path)
    detected = score > 0.5
    tp += int(detected)
    mark = 'OK' if detected else 'MISS'
    print(f'  {mark:4s} {wav_path.name:40s} {score:.4f}')

recall = tp / len(personal_wavs) if personal_wavs else 0
print(f'Recall: {recall:.0%} ({tp}/{len(personal_wavs)})')

# ── Test on adversarial negatives (should NOT detect) ────────────────
print('\n--- Adversarial negatives (expect no detection) ---')
neg_test_dir = os.path.join(OUTPUT_DIR, 'hey_vox', 'negative_test')
if os.path.exists(neg_test_dir):
    neg_wavs = sorted(Path(neg_test_dir).glob('*.wav'))[:200]
    fp = 0
    for wav_path in neg_wavs:
        score = test_detection(wav_path)
        if score > 0.5:
            fp += 1
    fpr = fp / len(neg_wavs) if neg_wavs else 0
    print(f'FPR: {fpr:.1%} ({fp}/{len(neg_wavs)})')
else:
    print(f'Negative test dir not found at {neg_test_dir}')
    fpr = -1

print(f'\n{"=" * 50}')
print(f'  Recall:  {recall:.0%}')
if fpr >= 0:
    print(f'  FPR:     {fpr:.1%}')
print(f'  Model:   {model_path}')
print(f'{"=" * 50}')

## 10. Download Model

Download `hey_vox.onnx` and place it at:
```
~/.config/heyvox/models/hey_vox.onnx
```

Then set in `~/.config/heyvox/config.yaml`:
```yaml
wake_word:
  start: hey_vox
  stop: hey_vox
```

In [ ]:
from google.colab import files
files.download(model_path)
print(f'\nDownloaded {model_path}')
print('Copy to: ~/.config/heyvox/models/hey_vox.onnx')